In [3]:
import cv2
import datetime
import pandas as pd
import numpy as np
from keras.models import load_model

In [5]:
from keras.models import load_model

# Don't compile when loading
age_model = load_model("/kaggle/input/age-gender-models/age_model.h5", compile=False)
gender_model = load_model("/kaggle/input/age-gender-models/gender_model.h5", compile=False)


In [6]:
# Load Haar cascade for face detection
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

In [7]:
# Load your video
video_path = "/kaggle/input/testvideo1/videoplayback.mp4"  # Update if needed
cap = cv2.VideoCapture(video_path)

In [8]:
log = []

In [18]:
while True:
    ret, frame = cap.read()
    if not ret:
        break

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray, 1.3, 5)

    for (x, y, w, h) in faces:
        face_img = frame[y:y+h, x:x+w]
        try:
            resized_face = cv2.resize(face_img, (224, 224)) / 255.0

        except:
            continue
        resized_face = np.expand_dims(resized_face, axis=0)

        # Predict age and gender
        age = int(age_model.predict(resized_face)[0][0])
        gender_pred = gender_model.predict(resized_face)
        gender = "Male" if gender_pred[0][0] > 0.5 else "Female"

        # If senior citizen
        if age > 60:
            time_stamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
            log.append({"Age": age, "Gender": gender, "Time": time_stamp})

cap.release()

In [19]:
# Save results to CSV
df = pd.DataFrame(log)
df.to_csv("/kaggle/working/senior_citizens_log1.csv", index=False)
print("✅ Detection complete. Log saved to /kaggle/working/senior_citizens_log1.csv")

✅ Detection complete. Log saved to /kaggle/working/senior_citizens_log1.csv


In [20]:
print(f"Detected {len(faces)} faces in frame")

Detected 1 faces in frame
